In [1]:
import masknmf
import torch
import sys
import numpy as np

from typing import *
import fastplotlib as fpl
import masknmf
import scipy
import h5py
import scipy.sparse
import numpy as np
import torch
import os
import roicat
from masknmf.multisession import RoicatDataAdapter, RoicatTracker, RoicatTrackingResults
from functools import partial

import tempfile

%load_ext autoreload
%matplotlib inline

W0827 16:09:10.363000 3620 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
To silence this warning, use a fully namespaced name.


In [9]:
from pathlib import Path

demix_path = Path(r"C:\Users\loson\data\eunji\masknmf-defaults\zplane01\demixing_results.hdf5")
demix_path.is_file()

True

In [13]:
# note this needs to be a list of paths
roicat_input = RoicatDataAdapter.from_masknmf([demix_path])

C:\Users\loson\repos\masknmf-toolbox\masknmf\utils\_serialization.py:191: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:767.)
  ans[key] = torch.sparse_coo_tensor(


Completed: Set FOV_height and FOV_width successfully.
Setting FOV_images...
Completed: Set FOV_images for 1 sessions successfully.
Completed: Created session_bool.
Completed: Set spatialFootprints for 1 sessions successfully.
Centroids must be set before ROI images can be created. Creating centroids now.
Completed: Created centroids.
Starting: Creating centered ROI images from spatial footprints...
Completed: Created ROI images.


In [14]:
roicat_input.ROI_images

[array([[[0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ...,
          0.0000000e+00, 0.0000000e+00, 0.0000000e+00],
         [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ...,
          0.0000000e+00, 0.0000000e+00, 0.0000000e+00],
         [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ...,
          0.0000000e+00, 0.0000000e+00, 0.0000000e+00],
         ...,
         [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ...,
          0.0000000e+00, 0.0000000e+00, 0.0000000e+00],
         [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ...,
          0.0000000e+00, 0.0000000e+00, 0.0000000e+00],
         [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ...,
          0.0000000e+00, 0.0000000e+00, 0.0000000e+00]],
 
        [[0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ...,
          0.0000000e+00, 0.0000000e+00, 0.0000000e+00],
         [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ...,
          0.0000000e+00, 0.0000000e+00, 0.0000000e+00],
         [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 

###  For each neuron, make random labels (0 or 1) just as a proof of concept. The GUI will replace this. Note that 

In [20]:
class_labels = []
print(type(roicat_input.n_roi))

for k in range(roicat_input.n_sessions):
    num_labels = roicat_input.n_roi[k]
    random_bin_vec = np.random.choice(2, size = num_labels)
    print(len(random_bin_vec))
    print(num_labels)
    class_labels.append(random_bin_vec)

<class 'list'>
1556
1556
1611
1611
1566
1566
1609
1609
1659
1659
1589
1589


In [21]:
roicat_input.set_class_labels(labels = class_labels)

Starting: Importing class labels
Labels and ROI Images match in shapes: Class labels and ROI images have the same number of sessions and the same number of ROIs in each session.
Completed: Imported labels for 6 sessions. Each session has [1556, 1611, 1566, 1609, 1659, 1589] class labels. Total number of class labels is 9590.


In [25]:
DEVICE = roicat.helpers.set_device(use_GPU=True, verbose=True)
dir_temp = tempfile.gettempdir()

roinet = roicat.ROInet.ROInet_embedder(
    device=DEVICE,  ## Which torch device to use ('cpu', 'cuda', etc.)
    dir_networkFiles=dir_temp,  ## Directory to download the pretrained network to
    download_method='check_local_first',  ## Check to see if a model has already been downloaded to the location (will skip if hash matches)
    download_url='https://osf.io/c8m3b/download',  ## URL of the model
    download_hash='357a8d9b630ec79f3e015d0056a4c2d5',  ## Hash of the model file
    forward_pass_version='head',  ## How the data is passed through the network
    verbose=True,  ## Whether to print updates
)

roinet.generate_dataloader(
    ROI_images=roicat_input.ROI_images,  ## Input images of ROIs
    um_per_pixel=roicat_input.um_per_pixel,  ## Resolution of FOV
    pref_plot=False,  ## Whether or not to plot the ROI sizes
);

Using device: cuda:0
File already exists locally: /tmp/ROInet.zip
Hash of local file matches provided hash_hex.
Extracting /tmp/ROInet.zip to /tmp.
Completed zip extraction.
Imported model from /tmp/ROInet_classification_20220902/model.py
Loaded params_model from /tmp/ROInet_classification_20220902/params.json


/data/home/app2139/masknmf-toolbox/.multisession_venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/data/home/app2139/masknmf-toolbox/.multisession_venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1`. You can also use `weights=ConvNeXt_Tiny_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Generated network using params_model
Loaded state_dict into network from /tmp/ROInet_classification_20220902/ConvNext_tiny__1_0_unfrozen__simCLR.pth
Loaded network onto device cuda:0
Starting Image Resizer
ROICaT: replacing NaNs with 0.0
ROICaT: resizing ROIs


  0%|          | 0/1556 [00:00<?, ?it/s]

ROICaT: replacing NaNs with 0.0
ROICaT: resizing ROIs


...

  0%|          | 0/1659 [00:00<?, ?it/s]

ROICaT: replacing NaNs with 0.0
ROICaT: resizing ROIs


  0%|          | 0/1589 [00:00<?, ?it/s]

Creating dataloader
Defined image transformations: Sequential(
  (0): ScaleDynamicRange(scaler_bounds=(0, 1))
  (1): Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
  (2): TileChannels(dim=-3)
)
Defined dataset
Defined dataloader


/data/home/app2139/masknmf-toolbox/.multisession_venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 24 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [26]:
roinet.generate_latents()

starting: running data through network


/data/home/app2139/masknmf-toolbox/.multisession_venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 24 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


  0%|          | 0/1199 [00:01<?, ?it/s]

completed: running data through network


tensor([[  -0.0000,   -0.0000,  465.5896,  ...,   -0.0000,   -0.0000,
          472.9560],
        [  -0.0000,   -0.0000,    5.5882,  ...,   -0.0000, 1506.4100,
         1134.5667],
        [  -0.0000,   -0.0000,  258.7503,  ...,   -0.0000, 1370.8540,
          395.3506],
        ...,
        [  -0.0000,   -0.0000,  437.5575,  ...,   -0.0000,   -0.0000,
         2962.5942],
        [  -0.0000,   -0.0000,   -0.0000,  ...,   -0.0000,   -0.0000,
         2281.1777],
        [  -0.0000,   -0.0000, 2302.2778,  ...,   -0.0000,   -0.0000,
         3799.6528]])

In [28]:
x = np.array(roinet.latents).astype(np.float32)
y = np.concatenate(roicat_input.class_labels_index).astype(np.int64)

/tmp/ipykernel_117809/377227988.py:1: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  x = np.array(roinet.latents).astype(np.float32)


In [31]:
autoclassifier = roicat.classification.classifier.Auto_LogisticRegression(
    X=x,
    y=y,
    params_LogisticRegression={
        'C': [1e-13, 1e3],
    },
    verbose=True,
)
autoclassifier.fit()

[I 2026-08-27 10:42:14,978] A new study created in memory with name: Autotuner


  0%|          | 0/150 [00:00<?, ?it/s]

[I 2026-08-27 10:43:05,780] Trial 0 finished with value: 0.7042431288555994 and parameters: {'C': 2.942514308918573e-05}. Best is trial 0 with value: 0.7042431288555994.
[I 2026-08-27 10:43:54,171] Trial 1 finished with value: 0.7233250990264144 and parameters: {'C': 5.731126048613479e-05}. Best is trial 0 with value: 0.7042431288555994.
[I 2026-08-27 10:44:51,931] Trial 2 finished with value: 0.6931449459829551 and parameters: {'C': 1.6228843897552016e-13}. Best is trial 2 with value: 0.6931449459829551.
[I 2026-08-27 10:45:35,243] Trial 3 finished with value: 0.6939876495128353 and parameters: {'C': 3.669139939954747e-11}. Best is trial 2 with value: 0.6931449459829551.
[I 2026-08-27 10:46:04,935] Trial 4 finished with value: 0.7028725065212548 and parameters: {'C': 5.495251706033474e-09}. Best is trial 2 with value: 0.6931449459829551.
[I 2026-08-27 10:46:55,700] Trial 5 finished with value: 0.7135291748544924 and parameters: {'C': 8.418638308396394}. Best is trial 2 with value: 0.6

(LogisticRegression(C=3.403667987557976e-12,
                    class_weight={np.int64(0): np.float64(0.9825819672131147),
                                  np.int64(1): np.float64(1.0180467091295118)}),
 {'C': 3.403667987557976e-12})

In [42]:
packet = roicat.classification.ClassifierPackage(
    classifier=autoclassifier,  ## The fitted Auto_LogisticRegression from above
    embedder=roinet,  ## The ROInet_embedder that produced the latents
    label_names=[str(l) for l in autoclassifier.model_best.classes_],  ## Class names, in classes_ order
    size_images_in=roicat_input.ROI_images[0].shape[1:],  ## (height, width) of the RAW ROI images
    um_per_pixel_training=roicat_input.um_per_pixel[0],  ## Recorded as provenance only
)
paths_save = os.path.abspath('./roicat_classifiers/roicat_result_1')
packet.save(paths_save, overwrite=True)
print(f"Saved packet to: {paths_save}")

Saved packet to: /data/home/app2139/masknmf-toolbox/2p_calcium_testsuite/may_2026_mutisession/roicat_classifiers/roicat_result_1
